In [ ]:
#@title Install ComfyUI { display-mode: "form" }
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFYUI = True  #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True  #@param {type:"boolean"}

import os, subprocess

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/drive/MyDrive/ComfyUI'
else:
    WORKSPACE = '/content/ComfyUI'

os.environ['COMFYUI_WORKSPACE'] = WORKSPACE  # used by later cells

if not os.path.exists(os.path.join(WORKSPACE, '.git')):
    !git clone https://github.com/comfyanonymous/ComfyUI '{WORKSPACE}'
elif UPDATE_COMFYUI:
    !git pull '{WORKSPACE}'

# Python dependencies. Colab already ships a CUDA build of PyTorch, so this
# mostly installs the lighter packages ComfyUI needs on top of it.
!pip install -q -r '{WORKSPACE}/requirements.txt'

# aria2 for fast, resumable model downloads in Cell 3
!apt-get -y -qq install aria2 > /dev/null

if INSTALL_COMFYUI_MANAGER:
    mgr = os.path.join(WORKSPACE, 'custom_nodes', 'ComfyUI-Manager')
    if not os.path.exists(mgr):
        !git clone https://github.com/ltdrdata/ComfyUI-Manager '{mgr}'
    else:
        !git pull '{mgr}'
    !pip install -q -r '{mgr}/requirements.txt'

print(f"\nComfyUI installed at: {WORKSPACE}")


In [ ]:
import os
import requests

path = os.path.join(WORKSPACE, "/models")
models_to_download = [ # For the first example. Add and remove as you like.
    ("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "vae"),
    ("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors", "diffusion_models"),
    ("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", "text_encoders"),
]

for url, subfolder in models_to_download:
    url = url.replace('/blob/', '/resolve/') # ComfyUI gives you 'blob' URLs by default, catch and fix this before it causes any problems.
    target_dir = os.path.join(path, subfolder)
    filename = url.split('/')[-1]
    target_file = os.path.join(target_dir, filename)
    os.makedirs(target_dir, exist_ok=True)
    if os.path.exists(target_file):
        print(f"{target_file} already exists")
    else:
        try:
            print(f"Downloading: {filename}")
            response = requests.get(url, stream=True, timeout=30)
            response.raise_for_status()  # Check for HTTP errors

            with open(target_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print(f"✓ Downloaded: {target_file}")
        except requests.exceptions.RequestException as e:
            print(f"✗ Error downloading {url}: {e}")
        except Exception as e:
            print(f"✗ Unexpected error: {e}")

In [ ]:
#@title Start server + tunnel { display-mode: "form" }
import os, socket, subprocess, threading, time, sys

WORKSPACE = os.environ.get('COMFYUI_WORKSPACE', '/content/ComfyUI')
PORT = 8188

# --- install cloudflared (quick tunnel, no account needed) ---
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True)

def tunnel_when_ready(port):
    # wait for the ComfyUI server to accept connections
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) == 0:
                break
        time.sleep(0.5)
    # --protocol http2 forces TCP transport; cloudflared's default QUIC (UDP)
    # is unreliable from Colab VMs and causes tunnels that half-work
    # (page loads, then assets/API/websocket stall).
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}",
         "--protocol", "http2"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    url = None
    for raw in p.stderr:
        line = raw.decode(errors="ignore")
        if "trycloudflare.com" in line and "https://" in line:
            url = line[line.find("https://"):].split()[0]
            break
    if url is None:
        print("cloudflared exited without issuing a URL -- rerun this cell.")
        return
    # End-to-end self-check: hit the public URL through Cloudflare's edge
    # from this VM before handing it to the user.
    import requests
    print(f"\nTunnel URL issued: {url}")
    print("Verifying it end-to-end through Cloudflare's edge...")
    ok = False
    for attempt in range(12):  # edge DNS can take ~30-60s to propagate
        try:
            r1 = requests.get(url, timeout=15)
            r2 = requests.get(url + "/api/system_stats", timeout=15)
            if r1.status_code == 200 and r2.status_code == 200:
                ok = True
                break
            print(f"  attempt {attempt+1}: page={r1.status_code} api={r2.status_code}, retrying...")
        except Exception as e:
            print(f"  attempt {attempt+1}: {type(e).__name__}, retrying...")
        time.sleep(5)
    if ok:
        print("\n" + "=" * 60)
        print("  VERIFIED WORKING -- open ComfyUI at:")
        print(" ", url)
        print("=" * 60 + "\n", flush=True)
    else:
        print("\nThe tunnel URL is not responding correctly through the edge.")
        print("Interrupt this cell and rerun it to get a fresh tunnel.\n", flush=True)

threading.Thread(target=tunnel_when_ready, args=(PORT,), daemon=True).start()

cmd = [sys.executable, "main.py", "--listen", "127.0.0.1",
       "--port", str(PORT), "--dont-print-server",
       # Required when serving through a tunnel: ComfyUI's default middleware
       # returns 403 for any request with Sec-Fetch-Site: cross-site (e.g.
       # clicking the tunnel link from the Colab page).
       "--enable-cors-header"]

subprocess.run(cmd, cwd=WORKSPACE)
